# Hybrid Retrieval in RAG

> **Combining semantic meaning with exact-term matching.**

In the previous tutorials, we built a semantic retriever and moved our vector storage into **Qdrant Cloud**.

Our retrieval pipeline currently looks like:

```text
User Query
    ↓
Query Embedding
    ↓
Vector Search
    ↓
Top-K Results
```

This works well when semantic meaning is the main signal.

But real-world search is not always like that.

Consider queries containing:

```text
NDPA
Qwen3.5-4B
REF-2026-00491
invoice-83921
Section 4.2
```

These are cases where exact terms can matter enormously.

In this tutorial, we'll explore why **dense retrieval** and **lexical retrieval** are complementary, then combine them into a hybrid retrieval pipeline.

## What We'll Learn

By the end of this tutorial, you'll understand:

- Why dense retrieval can miss useful results
- What lexical retrieval is
- The difference between semantic and lexical matching
- Why combining retrieval methods can improve recall
- Why raw similarity scores from different retrievers shouldn't simply be added
- How Reciprocal Rank Fusion (RRF) works
- How to implement RRF ourselves
- How hybrid retrieval combines candidate lists
- The strengths and limitations of hybrid retrieval

We'll first build the concepts with a small corpus before connecting the approach to our Qdrant-based RAG pipeline.

# 1. The Limitation of Dense Retrieval

Our current retriever uses embeddings.

Conceptually:

```text
Query
  ↓
Embedding
  ↓
Vector similarity
  ↓
Ranked chunks
```

This is powerful because semantically similar expressions can be matched even when they don't share the same words.

For example:

```text
Query:
How long can I get my money back?

Document:
Customers can request a refund within 30 days.
```

The wording is different, but the meaning is related.

However, exact terms can be important.

Consider:

```text
Query:
What does policy REF-2026-00491 say?
```

If the document contains the exact identifier:

```text
REF-2026-00491
```

we want retrieval to recognize that exact match.

This is where lexical retrieval becomes useful.

# 2. Dense Retrieval vs Lexical Retrieval

There are two useful ways to think about search.

### Dense retrieval

Dense retrieval asks:

> **Which documents have a similar meaning to this query?**

It works with embeddings.

```text
Query
  ↓
Embedding
  ↓
Semantic similarity
```

### Lexical retrieval

Lexical retrieval asks:

> **Which documents contain terms that match the query?**

It works with words or tokens.

```text
Query
  ↓
Terms
  ↓
Keyword matching
```

Neither approach is universally better.

They solve different problems.

```text
Dense
  → semantic relationships

Lexical
  → exact or term-based relationships
```

Hybrid retrieval combines both.

# 3. Our Search Corpus

Let's create a corpus designed to expose the difference between semantic and lexical retrieval.

In [1]:
documents = [
    {
        "id": "refund",
        "text": "Customers can request a refund within 30 days of purchase."
    },
    {
        "id": "refund-processing",
        "text": "Approved refunds are normally processed within 7 business days."
    },
    {
        "id": "policy-ndpa",
        "text": "The company processes personal data according to NDPA requirements."
    },
    {
        "id": "policy-ref",
        "text": "Policy REF-2026-00491 defines the company's customer data retention rules."
    },
    {
        "id": "invoice",
        "text": "Invoice INV-83921 was issued for the annual enterprise subscription."
    },
    {
        "id": "shipping",
        "text": "Standard shipping normally takes between 3 and 5 business days."
    },
]

# 4. Build the Dense Retriever

We'll use the same embedding model from our previous tutorials.

Because this notebook should be independently runnable, we install and load the model here.

In [2]:
! pip install -q sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
texts = [document["text"] for document in documents]

document_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

Now define a small dense-retrieval function.

In [5]:
def dense_retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    scores = document_embeddings @ query_embedding
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "id": documents[index]["id"],
            "text": documents[index]["text"],
            "score": float(scores[index]),
            "rank": rank,
        }
        for rank, index in enumerate(ranked_indices, start=1)
    ]

Test it with a semantic query:

In [6]:
query = "How long can a customer ask for their money back?"

dense_results = dense_retrieve(query)

for result in dense_results:
    print(f"Rank {result['rank']} | Score: {result['score']:.4f}")
    print(result["id"])
    print(result["text"])
    print("---")

Rank 1 | Score: 0.8165
refund
Customers can request a refund within 30 days of purchase.
---
Rank 2 | Score: 0.7189
refund-processing
Approved refunds are normally processed within 7 business days.
---
Rank 3 | Score: 0.6304
shipping
Standard shipping normally takes between 3 and 5 business days.
---


The important property is that the dense retriever can find conceptually related text even when the exact wording differs.

# 5. Build a Simple Lexical Retriever

Before using a production search engine, let's understand the basic idea ourselves.

We'll create a deliberately simple keyword retriever.

It will:

1. Tokenize the query.
2. Tokenize each document.
3. Count how many query terms appear in each document.
4. Rank documents by the number of matches.

This is **not** a production lexical search engine.

It is a learning implementation that lets us see what lexical retrieval is doing.

In [7]:
import re

def tokenize(text):
    return set(
        re.findall(r"[a-zA-Z0-9_-]+", text.lower())
    )

In [8]:
def lexical_retrieve(query, top_k=3):
    query_terms = tokenize(query)

    results = []

    for document in documents:
        document_terms = tokenize(document["text"])
        matches = query_terms.intersection(document_terms)

        results.append(
            {
                "id": document["id"],
                "text": document["text"],
                "score": len(matches),
                "matched_terms": sorted(matches),
            }
        )

    results.sort(
        key=lambda result: result["score"],
        reverse=True
    )

    return results[:top_k]

Now try an identifier-heavy query:

In [9]:
query = "What does policy REF-2026-00491 say?"

lexical_results = lexical_retrieve(query)

for rank, result in enumerate(lexical_results, start=1):
    print(f"Rank {rank} | Matches: {result['score']}")
    print(result["id"])
    print(result["text"])
    print(f"Matched terms: {result['matched_terms']}")
    print("---")

Rank 1 | Matches: 2
policy-ref
Policy REF-2026-00491 defines the company's customer data retention rules.
Matched terms: ['policy', 'ref-2026-00491']
---
Rank 2 | Matches: 0
refund
Customers can request a refund within 30 days of purchase.
Matched terms: []
---
Rank 3 | Matches: 0
refund-processing
Approved refunds are normally processed within 7 business days.
Matched terms: []
---


The lexical retriever has an important property:

```text
Exact term
    ↓
Strong signal
```

That can be useful for identifiers, names, codes, product numbers, and technical terminology.

# 6. Compare the Two Retrievers

Let's run the same query through both systems.

```python
query = "What does policy REF-2026-00491 say?"
```

Dense retrieval considers semantic similarity.

Lexical retrieval considers matching terms.

In [10]:
query = "What does policy REF-2026-00491 say?"

dense_results = dense_retrieve(query)
lexical_results = lexical_retrieve(query)

print("DENSE RETRIEVAL")
print("=" * 40)

for result in dense_results:
    print(f"{result['rank']}. {result['id']}")

print("\nLEXICAL RETRIEVAL")
print("=" * 40)

for rank, result in enumerate(lexical_results, start=1):
    print(f"{rank}. {result['id']}")

DENSE RETRIEVAL
1. policy-ref
2. refund
3. invoice

LEXICAL RETRIEVAL
1. policy-ref
2. refund
3. refund-processing


You may find that the two methods produce different rankings.

That is exactly what makes hybrid retrieval useful.

If two retrieval systems make different mistakes, combining them can produce a stronger candidate set.

# 7. Why We Shouldn't Simply Add the Scores

At first, we might try:

```text
hybrid_score = dense_score + lexical_score
```

But this is problematic.

Our dense scores might look like:

```text
0.81
0.76
0.72
```

while lexical scores might look like:

```text
3
2
1
```

These values are on different scales.

Adding them directly gives the numbers an arbitrary influence.

More generally:

> **Scores from different retrieval systems are not automatically comparable.**

We therefore need a way to combine rankings without requiring the raw scores to have the same scale.

That's where **Reciprocal Rank Fusion** comes in.

# 8. Reciprocal Rank Fusion

Reciprocal Rank Fusion, usually called **RRF**, combines ranked result lists.

A common formulation is:

```text
RRF(d) = Σ 1 / (k + rank(d))
```

where:

- `d` is a document
- `rank(d)` is its position in a result list
- `k` is a smoothing constant
- the contributions from multiple ranked lists are added together

The important idea is the behavior:

```text
Rank 1
  ↓
Strong contribution

Rank 10
  ↓
Smaller contribution
```

If a document appears highly in several retrieval lists, its combined score increases.

```text
Dense ranking
       +
Lexical ranking
       ↓
Higher combined score
```

# 9. Implement RRF

Let's implement the basic algorithm ourselves.

We'll use:

```python
k = 60
```

This is a common value, but it is a parameter that can be tuned.

In [16]:
from collections import defaultdict

def reciprocal_rank_fusion(result_lists, k=60):
    fused_scores = defaultdict(float)

    for results in result_lists:
        for rank, result in enumerate(results, start=1):
            fused_scores[result["id"]] += 1 / (k + rank)

    return sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

In [35]:
fused_scores = defaultdict(float)
for results in [dense_results, lexical_results]:
    for rank, result in enumerate(results, start=1):
        print(rank, result)
        fused_scores[result["id"]] += 1 / (60 + rank)
        print(fused_scores)
        print()
fused_scores

1 {'id': 'policy-ref', 'text': "Policy REF-2026-00491 defines the company's customer data retention rules.", 'score': 0.7907295227050781, 'rank': 1}
defaultdict(<class 'float'>, {'policy-ref': 0.01639344262295082})

2 {'id': 'refund', 'text': 'Customers can request a refund within 30 days of purchase.', 'score': 0.6138229966163635, 'rank': 2}
defaultdict(<class 'float'>, {'policy-ref': 0.01639344262295082, 'refund': 0.016129032258064516})

3 {'id': 'invoice', 'text': 'Invoice INV-83921 was issued for the annual enterprise subscription.', 'score': 0.6082643866539001, 'rank': 3}
defaultdict(<class 'float'>, {'policy-ref': 0.01639344262295082, 'refund': 0.016129032258064516, 'invoice': 0.015873015873015872})

1 {'id': 'policy-ref', 'text': "Policy REF-2026-00491 defines the company's customer data retention rules.", 'score': 2, 'matched_terms': ['policy', 'ref-2026-00491']}
defaultdict(<class 'float'>, {'policy-ref': 0.03278688524590164, 'refund': 0.016129032258064516, 'invoice': 0.015873

defaultdict(float,
            {'policy-ref': 0.03278688524590164,
             'refund': 0.03225806451612903,
             'invoice': 0.015873015873015872,
             'refund-processing': 0.015873015873015872})

Apply RRF to our dense and lexical result lists:

In [18]:
fused_results = reciprocal_rank_fusion(
    [dense_results, lexical_results]
)

for rank, (document_id, score) in enumerate(
    fused_results,
    start=1
):
    print(
        f"Rank {rank} | "
        f"ID: {document_id} | "
        f"RRF score: {score:.6f}"
    )

Rank 1 | ID: policy-ref | RRF score: 0.032787
Rank 2 | ID: refund | RRF score: 0.032258
Rank 3 | ID: invoice | RRF score: 0.015873
Rank 4 | ID: refund-processing | RRF score: 0.015873


RRF did **not** try to make the dense score and lexical score numerically compatible.

Instead, it used their **rank positions**.

That makes it useful for combining heterogeneous retrieval systems.

# 10. Turn the Fused IDs Back Into Documents

Our RRF function returns document IDs and fusion scores.

Let's recover the actual documents.

In [36]:
document_lookup = {
    document["id"]: document
    for document in documents
}

hybrid_results = [
    {
        "id": document_id,
        "score": score,
        "text": document_lookup[document_id]["text"],
    }
    for document_id, score in fused_results
]

for rank, result in enumerate(hybrid_results, start=1):
    print(f"Rank {rank} | RRF: {result['score']:.6f}")
    print(result["id"])
    print(result["text"])
    print("---")

Rank 1 | RRF: 0.032787
policy-ref
Policy REF-2026-00491 defines the company's customer data retention rules.
---
Rank 2 | RRF: 0.032258
refund
Customers can request a refund within 30 days of purchase.
---
Rank 3 | RRF: 0.015873
invoice
Invoice INV-83921 was issued for the annual enterprise subscription.
---
Rank 4 | RRF: 0.015873
refund-processing
Approved refunds are normally processed within 7 business days.
---


Our basic hybrid retriever is now:

```text
                     Query
                       │
              ┌────────┴────────┐
              ↓                 ↓
       Dense Retriever    Lexical Retriever
              ↓                 ↓
        Ranked List A       Ranked List B
              └────────┬────────┘
                       ↓
                      RRF
                       ↓
                Hybrid Ranking
```

# 11. Why Hybrid Retrieval Can Improve Recall

Imagine the correct document is:

```text
Document X
```

Dense retrieval ranks it:

```text
Rank 8
```

Lexical retrieval ranks it:

```text
Rank 1
```

If we only use dense retrieval with:

```text
top_k = 3
```

Document X never reaches the next stage.

But hybrid retrieval can promote it because the lexical retriever found a strong exact-term match.

This is one of the main motivations for hybrid retrieval:

> **Use different retrieval signals to increase the chance that useful evidence enters the candidate set.**

# 12. Hybrid Retrieval Is Still Retrieval

Hybrid retrieval doesn't solve every RAG problem.

It can still return:

- Irrelevant chunks
- Duplicates
- Incomplete context
- Outdated information
- Conflicting documents

For example:

```text
Dense Retrieval
       ↓
Relevant candidate

Lexical Retrieval
       ↓
Relevant candidate

Hybrid
       ↓
Still needs ranking
```

This is why hybrid retrieval is often followed by another stage.

We may retrieve a relatively large candidate set and then use a **reranker** to determine which results are actually the best evidence.

# 13. Hybrid Retrieval With Qdrant

In our previous notebook, Qdrant handled dense vector search.

A production architecture can now look like:

```text
                       Query
                         │
                ┌────────┴────────┐
                ↓                 ↓
           Qdrant Search     Lexical Search
                ↓                 ↓
          Dense Candidates    Lexical Candidates
                └────────┬────────┘
                         ↓
                        RRF
                         ↓
                 Hybrid Candidates
                         ↓
                    Reranker
                         ↓
                  Final Context
                         ↓
                        LLM
```

Qdrant is therefore still responsible for the dense retrieval side.

The simple keyword implementation in this notebook is for learning the concept. Production systems normally use a proper lexical index.

# 14. The Important Design Decision

It is tempting to conclude:

> "Hybrid retrieval is always better."

That's not quite right.

Every additional retrieval component adds:

- Complexity
- Infrastructure
- Latency
- Maintenance
- More parameters to tune

The right question is:

> **Does the additional retrieval signal solve a real failure mode in our application?**

For a corpus full of technical identifiers, names, codes, and exact terminology, lexical retrieval can be extremely valuable.

For another corpus, dense retrieval may already perform very well.

Good RAG engineering is about measuring these trade-offs rather than adding components because they are fashionable.

# 15. Debugging Hybrid Retrieval

When hybrid retrieval produces a poor result, inspect each stage independently.

Start with:

```text
Query
```

Then inspect:

```text
Dense results
```

Then:

```text
Lexical results
```

Then:

```text
RRF ranking
```

Then:

```text
Final candidates
```

This gives us a useful debugging path:

```text
Query
  ↓
Dense retrieval ──┐
                  ├──→ RRF → Candidates
Lexical retrieval ┘
```

If a relevant document appears in neither list, the problem is probably earlier in the retrieval design.

If it appears in one list but disappears from the final candidates, investigate the fusion or ranking stage.

# 16. The Hybrid Retrieval Mental Model

The key distinction to remember is:

```text
Dense Retrieval
    ↓
"What means something similar?"

Lexical Retrieval
    ↓
"What contains or matches these terms?"

Hybrid Retrieval
    ↓
"Can we use both signals?"
```

The goal isn't to make the retrieval system complicated.

The goal is to make the candidate set more useful.

# Key Takeaways

1. Dense retrieval is excellent at finding semantically similar information.
2. Lexical retrieval is useful when exact terms matter.
3. Identifiers, codes, names, and technical terminology can expose weaknesses in pure dense retrieval.
4. Dense and lexical retrieval often make different errors.
5. Combining them can improve the chance that relevant evidence enters the candidate set.
6. Raw scores from different retrievers should not automatically be added because they may use different scales.
7. Reciprocal Rank Fusion combines ranked lists without requiring their raw scores to be comparable.
8. Hybrid retrieval is still only a retrieval stage—it does not guarantee that the final context is correct.
9. Production systems should measure whether hybrid retrieval actually improves retrieval metrics.
10. More components mean more complexity, latency, and maintenance, so hybrid retrieval should solve a real problem.

The mental model is:

```text
                     Query
                       │
              ┌────────┴────────┐
              ↓                 ↓
            Dense            Lexical
          Retrieval          Retrieval
              ↓                 ↓
              └────────┬────────┘
                       ↓
                      RRF
                       ↓
               Candidate Set
                       ↓
                   Reranking
                       ↓
                    Context
                       ↓
                      LLM
```

# What's Next?

We now have a way to combine multiple retrieval signals.

But there is still a problem.

Suppose hybrid retrieval gives us:

```text
20 candidate chunks
```

Some are highly relevant.

Some are only loosely related.

Some may be irrelevant.

We don't necessarily want to send all 20 chunks to the LLM.

We need another mechanism that can look at the **query and each candidate together** and determine which candidates are actually the best matches.

That's the job of a **reranker**.

In the next tutorial:

> **Reranking: Turning a Broad Candidate Set Into High-Quality Context**

We'll explore why retrieval and reranking are separate stages, how cross-encoder rerankers work, and how reranking can improve the quality of the context we ultimately give to the LLM.